In [1]:
!git clone https://github.com/AIRI-Institute/SAE-Reasoning.git

Cloning into 'SAE-Reasoning'...
remote: Enumerating objects: 8589, done.
remote: Counting objects: 100% (3295/3295), done.
remote: Compressing objects: 100% (824/824), done.
remote: Total 8589 (delta 2786), reused 2471 (delta 2471), pack-reused 5294 (from 1)
Receiving objects: 100% (8589/8589), 10.76 MiB | 11.69 MiB/s, done.
Resolving deltas: 100% (5016/5016), done.


In [ ]:
%pip install -e SAE-Reasoning/TransformerLens sae-lens circuitsvis torch==2.4.1 torchvision==0.19.1 numpy==1.26.3

In [ ]:
from transformer_lens import HookedTransformer

In writeup, note that QwQ weights do not load in to most colab instances, even with high RAM.

In [ ]:
f

introduce fundamentals of modified transformerlens version, spending time on architectural changes.

In [5]:
import torch
import os

from sae_lens import (
    LanguageModelSAERunnerConfig,
    SAETrainingRunner,
    StandardTrainingSAEConfig,
    LoggingConfig,
    LanguageModelSAETrainingRunner,
)

if torch.cuda.is_available():
    device = "cuda"
elif torch.mps.device_count() > 1:
    device = "mps"
else:
  device = "cpu"

print("Using device:", device)

Using device: cuda


In [6]:
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["WANDB_API_KEY"] = "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"

In [7]:
total_training_steps = 30_000  # probably we should do more
batch_size = 4096

print(total_training_steps * batch_size)

122880000


In [8]:

total_training_steps = 30_000  # probably we should do more
batch_size = 4096
total_training_tokens = total_training_steps * batch_size

lr_warm_up_steps = 0
lr_decay_steps = total_training_steps // 5  # 20% of training
l1_warm_up_steps = total_training_steps // 20  # 5% of training

cfg = LanguageModelSAERunnerConfig(
    # Data Generating Function (Model + Training Distibuion)
    checkpoint_path="/content/drive/MyDrive/sae_checkpoints",
    model_name="deepseek-ai/DeepSeek-R1-Distill-Llama-8B",
    hook_name="blocks.19.hook_resid_post",
    dataset_path="andreuka18/DeepSeek-R1-Distill-Llama-8B-lmsys-openthoughts-tokenized",  # this is a tokenized language dataset on Huggingface for the Tiny Stories corpus.
    is_dataset_tokenized=True,
    model_from_pretrained_kwargs={
        "center_writing_weights": False,
        "dtype" : "bfloat16",
    },
    streaming=False,
    prepend_bos=False,# we could pre-download the token dataset if it was small.
    # SAE Parameters
    sae=StandardTrainingSAEConfig(
        d_in=4096,  # the width of the mlp output.
        d_sae=16384,  # the width of the SAE. Larger will result in better stats but slower training.
        apply_b_dec_to_input=False,  # We won't apply the decoder weights to the input.
        normalize_activations="expected_average_only_in",
        l1_coefficient=5,  # will control how sparse the feature activations are
        l1_warm_up_steps=l1_warm_up_steps,  # this can help avoid too many dead features initially.
    ),
    # Training Parameters
    lr=5e-5,  # lower the better, we'll go fairly high to speed up the tutorial.
    adam_beta1=0.9,  # adam params (default, but once upon a time we experimented with these.)
    adam_beta2=0.999,
    lr_scheduler_name="constant",  # constant learning rate with warmup. Could be better schedules out there.
    lr_warm_up_steps=0,  # this can help avoid too many dead features initially.
    lr_decay_steps=40000,  # this will help us avoid overfitting.
    train_batch_size_tokens=batch_size,
    context_size=1024,  # will control the lenght of the prompts we feed to the model. Larger is better but slower. so for the tutorial we'll use a short one.
    # Activation Store Parameters
    n_batches_in_buffer=32,  # controls how many activations we store / shuffle.
    training_tokens=total_training_tokens,  # 100 million tokens is quite a few, but we want to see good stats. Get a coffee, come back.
    store_batch_size_prompts=8,
    dead_feature_threshold=1e-4,  # would effect resampling or ghost grads if we were using it.
    # WANDB
    logger=LoggingConfig(
        log_to_wandb=True,  # always use wandb unless you are just testing code.
        wandb_project="sae_r1_distill_8b",
        wandb_id="tgggwoij",
        wandb_log_frequency=30,
        eval_every_n_wandb_logs=20,
    ),
    # Misc
    device=device,
    seed=42,
    from_pretrained_path="/content/drive/MyDrive/sae_checkpoints/deepseek_r1_8b/og6zaqqu/24580096",
    n_checkpoints=5,
    dtype="float32",
    exclude_special_tokens=[128000, 128001]
)

# sparse_autoencoder = SAETrainingRunner(cfg).run()

In [4]:

from google.colab import drive
import shutil, os, glob, threading, time

DRIVE_CHECKPOINT_DIR = "/content/drive/MyDrive/sae_checkpoints/deepseek_r1_8b"
os.makedirs(DRIVE_CHECKPOINT_DIR, exist_ok=True)
print(f"Checkpoints will sync to:\n   {DRIVE_CHECKPOINT_DIR}")

_synced: set[str] = set()

def sync_checkpoints_to_drive(local_dir: str = "checkpoints",
                               drive_dir: str = DRIVE_CHECKPOINT_DIR) -> None:
    """Copy any new checkpoint folders/files from local_dir → Drive."""
    if not os.path.exists(local_dir):
        return
    for item in os.listdir(local_dir):
        src = os.path.join(local_dir, item)
        dst = os.path.join(drive_dir, item)
        if src in _synced:
            continue
        try:
            if os.path.isdir(src):
                shutil.copytree(src, dst, dirs_exist_ok=True)
            else:
                shutil.copy2(src, dst)
            _synced.add(src)
            print(f"💾 Synced to Drive: {item}")
        except Exception as e:
            print(f"⚠️  Drive sync failed for {item}: {e}")

def _background_sync(interval_seconds: int = 120) -> None:
    """Poll for new checkpoints every `interval_seconds` and push to Drive."""
    while not _stop_sync.is_set():
        sync_checkpoints_to_drive()
        _stop_sync.wait(interval_seconds)

_stop_sync = threading.Event()
_sync_thread = threading.Thread(target=_background_sync,
                                args=(120,),       # sync every 2 min
                                daemon=True)
_sync_thread.start()
print("🔄 Background Drive-sync thread started (every 2 min).")


batch_size = 4096
total_training_tokens = total_training_steps * batch_size

lr_warm_up_steps = 0
lr_decay_steps = total_training_steps // 5
l1_warm_up_steps = total_training_steps // 20

cfg = LanguageModelSAERunnerConfig(
    model_name="Qwen/QwQ-32B-Preview",
    hook_name="blocks.19.hook_resid_post",
    dataset_path="andreuka18/DeepSeek-R1-Distill-Llama-8B-lmsys-openthoughts-tokenized",
    is_dataset_tokenized=True,
    model_from_pretrained_kwargs={
        "center_writing_weights": False,
        "dtype": "bfloat16",
    },
    streaming=False,
    prepend_bos=False,
    sae=StandardTrainingSAEConfig(
        d_in=4096,
        d_sae=16384,
        apply_b_dec_to_input=False,
        normalize_activations="expected_average_only_in",
        l1_coefficient=5,
        l1_warm_up_steps=l1_warm_up_steps,
    ),
    lr=5e-5,
    adam_beta1=0.9,
    adam_beta2=0.999,
    lr_scheduler_name="constant",
    lr_warm_up_steps=0,
    lr_decay_steps=40000,
    train_batch_size_tokens=batch_size,
    context_size=1024,
    n_batches_in_buffer=32,
    training_tokens=total_training_tokens,
    store_batch_size_prompts=8,
    dead_feature_threshold=1e-4,
    logger=LoggingConfig(
        log_to_wandb=True,
        wandb_project="sae_r1_distill_8b",
        wandb_log_frequency=30,
        eval_every_n_wandb_logs=20,
    ),
    device=device,
    seed=42,
    n_checkpoints=5,
    checkpoint_path="checkpoints",
    dtype="float32",
    exclude_special_tokens=[128000, 128001],
)

# # ── Run training ──────────────────────────────────────────────────────────────
# try:
#     sparse_autoencoder = SAETrainingRunner(cfg).run()
# finally:
#     # Stop the background thread and do one final sync when training ends
#     _stop_sync.set()
#     print("\n⏳ Performing final checkpoint sync to Drive...")
#     sync_checkpoints_to_drive()
#     print("✅ All checkpoints synced to Google Drive.")

✅ Drive mounted. Checkpoints will sync to:
   /content/drive/MyDrive/sae_checkpoints/deepseek_r1_8b
🔄 Background Drive-sync thread started (every 2 min).


In [2]:
%pip install sae-lens transformer-lens

  Using cached sae_lens-6.44.2-py3-none-any.whl.metadata (6.8 kB)
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 145.1/145.1 kB 16.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 311.0/311.0 kB 34.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 977.7/977.7 kB 74.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.4/56.4 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.6/42.6 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.4/52.4 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 274.1/274.1 kB 34.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 236.9/236.9 kB 30.1 MB/s eta 0:00:00
  Created wheel for transformers-stream-generator: filename=transformers_stream_generator-0.0.5-py3-none-any.whl size=12426 sha256=bb4c48b442db6f283eee549f218a293f949e356e5dd4b1953206d8422a910a33
  Stored in direc

In [9]:
runner = LanguageModelSAETrainingRunner(
    cfg=cfg,
)

# 4. Resume training
trained_sae = runner.run()

ValueError: deepseek-ai/DeepSeek-R1-Distill-Llama-8B not found. Valid official model names (excl aliases): ['01-ai/Yi-34B', '01-ai/Yi-34B-Chat', '01-ai/Yi-6B', '01-ai/Yi-6B-Chat', 'ai-forever/mGPT', 'allenai/OLMo-1B-hf', 'allenai/OLMo-2-0425-1B', 'allenai/OLMo-2-1124-7B', 'allenai/Olmo-3-32B-Think', 'allenai/Olmo-3-7B-Instruct', 'allenai/Olmo-3-7B-Think', 'allenai/Olmo-3.1-32B-Instruct', 'allenai/Olmo-3.1-32B-Think', 'allenai/OLMo-7B-hf', 'allenai/OLMoE-1B-7B-0924', 'ArthurConmy/redwood_attn_2l', 'Baidicoot/Othello-GPT-Transformer-Lens', 'bigcode/santacoder', 'bigscience/bloom-1b1', 'bigscience/bloom-1b7', 'bigscience/bloom-3b', 'bigscience/bloom-560m', 'bigscience/bloom-7b1', 'codellama/CodeLlama-7b-hf', 'codellama/CodeLlama-7b-Instruct-hf', 'codellama/CodeLlama-7b-Python-hf', 'distilgpt2', 'EleutherAI/gpt-j-6B', 'EleutherAI/gpt-neo-1.3B', 'EleutherAI/gpt-neo-125M', 'EleutherAI/gpt-neo-2.7B', 'EleutherAI/gpt-neox-20b', 'EleutherAI/pythia-1.4b', 'EleutherAI/pythia-1.4b-deduped', 'Eleut